# Phase 4 : Préparation des données pour l'IA

**Objectif :** Transformer les données textuelles en une matrice numérique compréhensible par le modèle.

## Étapes :
1. Créer la matrice utilisateur-film (pivot table)
2. Remplir les valeurs manquantes
3. Convertir en matrice sparse
4. Diviser en jeu d'entraînement et de test

In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split

# Chargement
movies = pd.read_csv('../data/movies_clean.csv')
ratings = pd.read_csv('../data/ratings_clean.csv')

print('✅ Données chargées')

In [ ]:
# 1. MATRICE UTILISATEUR-FILM
user_movie_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
)

print(f'Dimensions de la matrice : {user_movie_matrix.shape}')
print(f'Utilisateurs : {user_movie_matrix.shape[0]}')
print(f'Films : {user_movie_matrix.shape[1]}')
print(f'Cellules non-nulles : {user_movie_matrix.notna().sum().sum():,}')

print('\nExtrait (5x5) :')
display(user_movie_matrix.iloc[:5, :5])

In [ ]:
# 2. REMPLISSAGE DES VALEURS MANQUANTES
# On remplit les NaN avec 0 (film non noté)
user_movie_filled = user_movie_matrix.fillna(0)

# Vérification
n_nan = user_movie_matrix.isna().sum().sum()
print(f'Valeurs manquantes initiales : {n_nan:,}')
print(f'Valeurs manquantes après remplissage : {user_movie_filled.isna().sum().sum()}')
print(f'Sparsité : {(1 - user_movie_filled.astype(bool).sum().sum() / user_movie_filled.size) * 100:.2f}%')

In [ ]:
# 3. MATRICE SPARSE (économie de mémoire)
sparse_matrix = csr_matrix(user_movie_filled.values)

size_dense = user_movie_filled.values.nbytes / 1e6
size_sparse = sparse_matrix.data.nbytes / 1e6
print(f'Taille matrice dense : {size_dense:.2f} MB')
print(f'Taille matrice sparse : {size_sparse:.2f} MB')
print(f'Économie mémoire : {(1 - size_sparse/size_dense) * 100:.1f}%')

In [ ]:
# 4. DIVISION TRAIN/TEST
train_data, test_data = train_test_split(
    ratings, test_size=0.2, random_state=42, stratify=ratings['userId']
)

print(f'Taille entraînement : {len(train_data):,}')
print(f'Taille test : {len(test_data):,}')
print(f'Ratio : {len(train_data)/len(ratings):.0%}/{len(test_data)/len(ratings):.0%}')

In [ ]:
# Sauvegarde
import joblib
user_movie_filled.to_pickle('../data/user_movie_matrix.pkl')
train_data.to_csv('../data/train_data.csv', index=False)
test_data.to_csv('../data/test_data.csv', index=False)

print('✅ Données préparées sauvegardées')